In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# I-configure ang noise management gamit ang Estimator

<Accordion>
<AccordionItem title="Mga bersyon ng package">

Ang code sa pahinang ito ay binuo gamit ang mga sumusunod na kinakailangan.
Inirerekumenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Mayroong ilang paraan para pamahalaan ang noise, karaniwang sa pamamagitan ng paggamit ng iba't ibang mga teknik ng error mitigation at error suppression para maiwasan ang mga error bago pa man mangyari. Ang mga teknik na ito ay karaniwang nagdudulot ng overhead sa pre-processing. Samakatuwid, mahalaga na makamit ang balanse sa pagitan ng pagperpekto ng iyong mga resulta at pagtitiyak na makukumpleto ang iyong job sa makatwirang oras.

Sinusuportahan ng Estimator ang mga sumusunod na teknik ng noise management. Tingnan ang [Mga teknik ng error mitigation at suppression](error-mitigation-and-suppression-techniques) para sa paliwanag ng bawat isa. Tingnan ang seksyon ng [Mga custom na setting ng error](#advanced-error) para sa mga tagubilin para i-enable ang mga teknik na ito.

- [dynamical decoupling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [Pauli twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## Antas ng resilience
Tinutukoy ng `resilience_level` kung gaano karaming resilience ang itatayo laban sa mga error. Ang mas mataas na antas ay nagbubuo ng mas tumpak na mga resulta, sa gastos ng mas matagal na oras ng pagproseso. Ang mga antas ng resilience ay maaaring gamitin para i-configure ang trade-off ng gastos/katumpakan kapag nag-aaplay ng noise management sa iyong primitive query. Binabawasan ng noise management ang mga error (bias) sa mga resulta sa pamamagitan ng pagproseso ng mga output mula sa isang koleksyon, o ensemble, ng mga kaugnay na circuit. Ang antas ng pagbabawas ng error ay depende sa pamamaraang inilapat. Ang antas ng resilience ay nag-aabstract ng detalyadong pagpili ng pamamaraan ng noise management para payagan ang mga gumagamit na mag-isip tungkol sa trade-off ng gastos/katumpakan na angkop sa kanilang aplikasyon.

Dahil dito, ang bawat antas ay tumutugma sa isang pamamaraan o mga pamamaraan na may tumataas na antas ng quantum sampling overhead para payagan kang mag-eksperimento sa iba't ibang trade-off ng oras-katumpakan. Ipinapakita sa sumusunod na talahanayan kung anong mga antas at kaukulang pamamaraan ang available para sa bawat isa sa mga primitive.

<span id="resilience-table"></span>

| Antas ng Resilience | Paglalarawan | Teknik |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | Walang mitigation                                                                                     | Wala                                                                      |
| 1 [Default]      | Minimal na gastos sa mitigation: Bawasan ang error na nauugnay sa mga error sa readout               | [Twirled Readout Error eXtinction (TREX)](/guides/error-mitigation-and-suppression-techniques#trex) measurement twirling             |
| 2                | Medium na gastos sa mitigation. Karaniwang binabawasan ang bias sa mga estimator, ngunit hindi ginagarantiyahan na zero-bias. | Antas 1 + [Zero Noise Extrapolation (ZNE)](/guides/error-mitigation-and-suppression-techniques#zne) at gate twirling

> **Info:** Ang mga antas ng resilience ay kasalukuyang nasa beta kaya't ang sampling overhead at kalidad ng solusyon ay mag-iiba-iba mula sa circuit patungong circuit. Ang mga bagong feature, advanced na opsyon, at mga tool sa pamamahala ay ilalabas sa rolling basis. Hindi ginagarantiyahan na ang mga tiyak na pamamaraan ng noise management ay ina-apply sa bawat antas ng resilience.

### I-configure ang Estimator gamit ang mga antas ng resilience
Maaari kang gumamit ng mga antas ng resilience para tukuyin ang mga teknik ng noise management, o maaari kang magtakda ng mga custom na teknik nang isa-isa tulad ng inilarawan sa [Mga custom na setting ng error](#advanced-error).

> **Note:** Anumang mga opsyon na manu-manong itinakda mo bilang karagdagan sa antas ng resilience ay inilalapat bilang karagdagan sa base na set ng mga opsyon na tinukoy ng antas ng resilience. Samakatuwid, sa prinsipyo, maaari mong itakda ang antas ng resilience sa 1, ngunit pagkatapos ay i-off ang measurement mitigation, bagaman hindi ito inirerekumenda.
> 
> Halimbawa, ang pagtatakda ng antas ng resilience sa 0 ay nagpapatay ng `zne_mitigation`, ngunit ang `estimator.options.resilience.zne_mitigation = True` ay nag-o-override ng halagang iyon.

### Halimbawa
Ang sumusunod na code ay nagpapagana ng ZNE, TREX, at gate twirling sa pamamagitan ng pagtatakda ng `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Mga custom na setting ng noise management
Maaari mong i-on at i-off ang mga indibidwal na pamamaraan ng noise management sa pamamagitan ng paggamit ng [mga opsyon ng Estimator](/guides/estimator-options).

> **Note:** Hindi lahat ng opsyon ay gumagana nang magkasama sa lahat ng uri ng circuit. Tingnan ang [feature compatibility table](estimator-options#options-compatibility-table) para sa mga detalye.

### Halimbawa

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: "
    f"{estimator.options.twirling.enable_gates}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>